In [1]:
!pip -q install torchvision

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.utils import save_image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


In [2]:
batch_size = 128
image_size = 28

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # [-1,1]
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 17.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 519kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.80MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.74MB/s]


In [3]:
z_dim = 100

class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, 1024),
            nn.ReLU(True),
            nn.Linear(1024, 28*28),
            nn.Tanh()  # output in [-1,1]
        )

    def forward(self, z):
        x = self.net(z)
        return x.view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.LeakyReLU(0.2, True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, True),
            nn.Linear(256, 1)  # logits
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.net(x)

G = Generator(z_dim).to(DEVICE)
D = Discriminator().to(DEVICE)

In [4]:
lr = 2e-4
epochs = 5  # increase to 20+ for better quality

optG = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
optD = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
bce = nn.BCEWithLogitsLoss()

os.makedirs("gan_samples", exist_ok=True)

for ep in range(1, epochs+1):
    for real, _ in train_loader:
        real = real.to(DEVICE)

        # -------------------
        # Train Discriminator
        # -------------------
        z = torch.randn(batch_size, z_dim, device=DEVICE)
        fake = G(z).detach()

        real_logits = D(real)
        fake_logits = D(fake)

        lossD = 0.5 * (
            bce(real_logits, torch.ones_like(real_logits)) +
            bce(fake_logits, torch.zeros_like(fake_logits))
        )

        optD.zero_grad()
        lossD.backward()
        optD.step()

        # -------------
        # Train Generator
        # -------------
        z = torch.randn(batch_size, z_dim, device=DEVICE)
        fake = G(z)
        fake_logits = D(fake)

        lossG = bce(fake_logits, torch.ones_like(fake_logits))

        optG.zero_grad()
        lossG.backward()
        optG.step()

    print(f"Epoch {ep}/{epochs} | lossD={lossD.item():.4f} | lossG={lossG.item():.4f}")

    # save a sample grid each epoch
    with torch.no_grad():
        z = torch.randn(64, z_dim, device=DEVICE)
        sample = G(z).cpu()
        save_image((sample + 1)/2, f"gan_samples/epoch_{ep:02d}.png", nrow=8)

print("✅ Training done. Check gan_samples/ folder.")

Epoch 1/5 | lossD=0.4481 | lossG=1.8702
Epoch 2/5 | lossD=0.2886 | lossG=2.6323
Epoch 3/5 | lossD=0.5431 | lossG=7.9301
Epoch 4/5 | lossD=0.8695 | lossG=2.4711
Epoch 5/5 | lossD=0.5283 | lossG=6.7799
✅ Training done. Check gan_samples/ folder.


In [5]:
import torchvision

os.makedirs("synthetic_mnist", exist_ok=True)

num_to_generate = 2000
batch_gen = 200

G.eval()
count = 0

with torch.no_grad():
    while count < num_to_generate:
        cur = min(batch_gen, num_to_generate - count)
        z = torch.randn(cur, z_dim, device=DEVICE)
        fake = G(z).cpu()                # [-1,1]
        fake = (fake + 1) / 2            # [0,1]

        for i in range(cur):
            save_image(fake[i], f"synthetic_mnist/fake_{count+i:05d}.png")
        count += cur

print("✅ Saved", num_to_generate, "fake images in synthetic_mnist/")

✅ Saved 2000 fake images in synthetic_mnist/
